In [ ]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

import utils
importlib.reload(utils)
from utils import *

import config
importlib.reload(config)
from config import *

##### This script:
- Creates and saves plots under the 'visuals' folder in each participant's folder
- Creates an excel file with the station stats and saves in each participant's folder

##### Before running
- Update the `config.py` file with the following information:
  - Folder paths :(base_dir) and make sure the participant data are under participant folders in your directory (format ex: P01)
  - Participant ID
  - Data file names
  - ActLumus timeshift
  - Study start and end times
  - Station timings
  - Participant groups mapped to study dates

In [ ]:
os.makedirs(f"{base_dir}/{save_id}/visuals", exist_ok=True)

---

In [ ]:
participant_studydate = pd.DataFrame(PARTICIPANT_MAP).T.reset_index()
participant_studydate.columns = ["participant_id", "date", "stations"]
participant_studydate = participant_studydate.sort_values(by='participant_id', ignore_index=True)

In [ ]:
participant_row = participant_studydate.loc[participant_studydate['participant_id'] == save_id].iloc[0]

# Automatically fill variables
date_str = participant_row['date']
save_date = date_str.replace("-", "")  # for filename
stations = participant_row['stations']
#-------------------------------------
start_interval = pd.to_datetime(f"{date_str} {start}")
end_interval = pd.to_datetime(f"{date_str} {end}")


In [ ]:
# statition windows : trimmed from the start and the end of the station times if needed
station_windows = {
    name: (
        pd.to_datetime(f"{date_str} {start}") + pd.Timedelta(seconds=0),
        pd.to_datetime(f"{date_str} {end}") - pd.Timedelta(seconds=0),
    )
    for name, (start, end) in stations.items()
}

# full station windows for the graph shadings
station_windows_full = {
    name: (
        pd.to_datetime(f"{date_str} {start}"),
        pd.to_datetime(f"{date_str} {end}")
    )
    for name, (start, end) in stations.items()
}

In [ ]:
actlumus_path = f"{base_dir}/{save_id}/{actlumus_file}"

with open(actlumus_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for i, line in enumerate(lines[:50]):  # check first 50 lines to see where the data starts
    if not line.startswith('#') and ';' in line:
        print(f"Actlumus data starts at line {i}: {line}")
        actlumus_start_line = i
        break
#-----------------------------------------------------
if iButton_file_chest is not None:
    ibutton_chest_path = f"{base_dir}/{save_id}/{iButton_file_chest}"

    with open(ibutton_chest_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for i, line in enumerate(lines[:50]):  # check first 50 lines to see where the data starts
        if line.startswith('\n'):
            print(f"iButton chest data starts at line {i+1}")
            ibutton_chest_start_line = i+1
            break
#-----------------------------------------------------
if iButton_file_hand is not None:
    ibutton_hand_path = f"{base_dir}/{save_id}/{iButton_file_hand}"

    with open(ibutton_hand_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for i, line in enumerate(lines[:50]):  # check first 50 lines to see where the data starts
        if line.startswith('\n'):
            print(f"iButton hand data starts at line {i+1}")
            ibutton_hand_start_line = i+1
            break


In [ ]:
hr = pd.read_csv(f"{base_dir}/{save_id}/{hr_file}")
act = pd.read_csv(actlumus_path, sep=';', header=0, skiprows=actlumus_start_line, engine='python')
hrv = pd.read_csv(f"{base_dir}/{save_id}/{hrv_file}")
hobo = pd.read_csv(f"{base_dir}/{save_id}/{hobo_file}", skiprows=1)

# MSR data read
columns = ["TIME", "ACC x", "ACC y", "ACC z", "RH", "T_RH", "L1", "A1", "A2", "BAT"]# Column names from *CHANNEL line (line 15)
msr = pd.read_csv(f"{base_dir}/{save_id}/{msr_file}",sep=';',encoding='latin1',skiprows=27,names=columns,engine='python')# skip everything including *DATA line --> 27 lines

if iButton_file_chest is not None:
    ibutton_chest = pd.read_csv(ibutton_chest_path, sep=',', header=None, names=["Date", "Time", "Unit", "Value", "Decimal"], skiprows=ibutton_chest_start_line, engine='python')
if iButton_file_hand is not None:
    ibutton_hand = pd.read_csv(ibutton_hand_path, sep=',', header=None, names=["Date", "Time", "Unit", "Value", "Decimal"], skiprows=ibutton_hand_start_line, engine='python')


---
#### HR
---

In [ ]:
#----------------------------------------
#extract HR start time for HRV matching 
#----------------------------------------

meta = hr.iloc[0]
date = meta["Date"]
start_time = meta["Start time"]

start_datetime = pd.to_datetime(
    f"{date} {start_time}",
    dayfirst=True
)

In [ ]:
#----------------------------------------
#clean and format hr df
#----------------------------------------
ts_columns = hr.iloc[1].tolist()
hr_clean = hr.iloc[2:].copy()
hr_clean.columns = ts_columns
hr_clean["Time"] = pd.to_timedelta(hr_clean["Time"], errors="coerce")
hr_clean["HR (bpm)"] = pd.to_numeric(hr_clean["HR (bpm)"], errors="coerce")
hr_clean["timestamp"] = start_datetime + hr_clean["Time"]
hr_clean = (
    hr_clean[["timestamp", "HR (bpm)"]]
    .rename(columns={"HR (bpm)": "hr"})
    .dropna()
    .sort_values("timestamp")
)

hr_plot = hr_clean[(hr_clean['timestamp'] >= start_interval) & 
                   (hr_clean['timestamp'] <= end_interval)]

In [ ]:
#----------------------------------------
# Computing HR stats at the stations 
#----------------------------------------
hr_station = []

for name, (start, end) in station_windows.items():
    tmp = hr_clean[
        (hr_clean["timestamp"] >= start) &
        (hr_clean["timestamp"] <= end)
    ].copy()

    tmp["station"] = name
    hr_station.append(tmp)

hr_station = pd.concat(hr_station, ignore_index=True)

hr_stats = hr_station.groupby("station").agg(
    hr_min=("hr", "min"),
    hr_max=("hr", "max"),
    hr_mean=("hr", "mean"),
    hr_sd=("hr", "std")
)
hr_stats["hr_cv"] = hr_stats["hr_sd"] / hr_stats["hr_mean"]
hr_stats.reset_index(inplace=True)

In [ ]:
#----------------------------------------
# Plot HR with stations shaded
#----------------------------------------

plt.figure(figsize=(14, 5))

# HR time series
plt.plot(hr_plot["timestamp"], hr_plot["hr"], 
         color="blue", linewidth=0.4, label="Heart Rate")
# Station shading 
for start, end in station_windows_full.values():
    plt.axvspan(start, end, color="silver", alpha=0.25)

ax = plt.gca()

# Show only HH:MM 
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# X axis ticks every 10 minutes
ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0, 60, 10)))

plt.xticks(rotation=45, ha='right')
plt.xlabel("Time")
plt.ylabel("Heart Rate")
plt.title("Heart Rate over Time with Station Periods Shaded")
plt.legend()
for spine in ax.spines.values():
    spine.set_linewidth(0.5)
plt.tight_layout()
plt.savefig(
    os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_HR_{save_date}.png"),
    dpi=400,
    bbox_inches="tight",
    facecolor="white"
)
plt.savefig(
    os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_HR_{save_date}.pdf"),
    bbox_inches="tight"
)
plt.show()

---
#### ActLumus
---

In [ ]:
#----------------------------------------
#clean and format actlumus df
#----------------------------------------
act['DATE/TIME'] = pd.to_datetime(act['DATE/TIME'], dayfirst=True, errors='coerce')

#Correct the device clock offset (set in config)

time_shift = pd.Timedelta(minutes=actlumus_shift_min, seconds=actlumus_shift_sec)
act['DATE/TIME'] = act['DATE/TIME'] - time_shift
actlumus = act[actlumus_cols_keep].copy()
actlumus = actlumus[actlumus['DATE/TIME'].dt.date == pd.to_datetime(date_str).date()]

act_plot = actlumus[(actlumus['DATE/TIME'] >= start_interval) & 
                   (actlumus['DATE/TIME'] <= end_interval)]

In [ ]:
#----------------------------------------
# Computing ActLumus stats at the stations 
#----------------------------------------

actlumus_stats_list = []

for station, (start_time, end_time) in station_windows.items():
    subset = actlumus[(actlumus['DATE/TIME'] >= start_time) & (actlumus['DATE/TIME'] <= end_time)]
    
    # Calculate mean, std, CV for each column
    actlumus_statistics = {
        'station': station,
        'light_min': subset['LIGHT'].min(),
        'light_max': subset['LIGHT'].max(),
        'light_mean': subset['LIGHT'].mean(),
        'light_std': subset['LIGHT'].std(),
        'light_cv': (subset['LIGHT'].std() / subset['LIGHT'].mean()) if subset['LIGHT'].mean() != 0 else np.nan,
        'melanopic_min': subset['MELANOPIC EDI'].min(),
        'melanopic_max': subset['MELANOPIC EDI'].max(),
        'melanopic_mean': subset['MELANOPIC EDI'].mean(),
        'melanopic_std': subset['MELANOPIC EDI'].std(),
        'melanopic_cv': (subset['MELANOPIC EDI'].std() / subset['MELANOPIC EDI'].mean()) if subset['MELANOPIC EDI'].mean() != 0 else np.nan,
        'temp_min': subset['TEMPERATURE'].min(),
        'temp_max': subset['TEMPERATURE'].max(),
        'temp_mean': subset['TEMPERATURE'].mean(),
        'temp_std': subset['TEMPERATURE'].std(),
        'temp_cv': (subset['TEMPERATURE'].std() / subset['TEMPERATURE'].mean()) if subset['TEMPERATURE'].mean() != 0 else np.nan,
        'ir_min': subset['IR LIGHT'].min(),
        'ir_max': subset['IR LIGHT'].max(),
        'ir_mean': subset['IR LIGHT'].mean(),
        'ir_std': subset['IR LIGHT'].std(),
        'ir_cv': (subset['IR LIGHT'].std() / subset['IR LIGHT'].mean()) if subset['IR LIGHT'].mean() != 0 else np.nan,
    }
    actlumus_stats_list.append(actlumus_statistics)

actlumus_stats = pd.DataFrame(actlumus_stats_list)

In [ ]:
#----------------------------------------
# Plot ActLumus with stations shaded
#----------------------------------------
for i, var in enumerate(actlumus_variables, 1):
    plt.figure(figsize=(12, 4))
    plt.plot(act_plot['DATE/TIME'], act_plot[var], color='black', linewidth=0.4, label=var)
    
    # Shade station windows
    for start, end in station_windows_full.values():
        plt.axvspan(start, end, color='silver', alpha=0.25)
    
    ax = plt.gca()

    # Show only HH:MM 
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

    # X axis ticks every 10 minutes
    ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0, 60, 10)))

    plt.xticks(rotation=45, ha='right')
    plt.ylabel(actlumus_y_labels[i-1])
    plt.title(f"{var} over Time with Station Periods Shaded")
    plt.legend()

    plt.xlabel("Time")

    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    plt.tight_layout()
    plt.savefig(
        os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_{var}_{save_date}_timeadjusted.png"),
        dpi=400,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.savefig(
    os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_{var}_{save_date}_timeadjusted.pdf"),
    bbox_inches="tight"
)
    plt.show()

---
#### HRV
---

In [ ]:
#----------------------------------------
#clean and format hrv df
#----------------------------------------
hrv = hrv[["duration"]].copy()
start_time = start_datetime #comes from hr file

hrv['timestamp'] = start_time + pd.to_timedelta(hrv['duration'].cumsum(), unit='ms')
hrv_plot = hrv[(hrv['timestamp'] >= start_interval) & 
                   (hrv['timestamp'] <= end_interval)]

In [ ]:
#----------------------------------------
#Computing HRV stats at the stations 
#----------------------------------------
hrv_stats_list = []

for station, (start_time, end_time) in station_windows.items():
    subset = hrv[(hrv['timestamp'] >= start_time) & (hrv['timestamp'] <= end_time)]
    
    stats = {
        'station': station,
        'rr_min': subset['duration'].min(),
        'rr_max': subset['duration'].max(),
        'rr_mean': subset['duration'].mean(),
        'rr_std': subset['duration'].std(),
        'rr_cv': (subset['duration'].std() / subset['duration'].mean()) if subset['duration'].mean() != 0 else np.nan
    }
    hrv_stats_list.append(stats)

hrv_stats = pd.DataFrame(hrv_stats_list)
print(hrv_stats)

In [ ]:
#----------------------------------------
# Plot HRV with stations shaded
#----------------------------------------
plt.figure(figsize=(14,5))
plt.plot(hrv_plot['timestamp'], hrv_plot['duration'], color='black', linewidth=0.25, label='RR interval')

# Shade station windows
for start, end in station_windows_full.values():
    plt.axvspan(start, end, color='silver', alpha=0.25)
ax = plt.gca()

# Show only HH:MM 
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# X axis ticks every 10 minutes
ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0, 60, 10)))

plt.xticks(rotation=45, ha='right')
plt.xlabel("Time")
plt.ylabel("RR Interval (ms)")
plt.title("RR Intervals over Time with Station Windows Shaded")
plt.legend()
for spine in ax.spines.values():
    spine.set_linewidth(0.5)
plt.tight_layout()
plt.savefig(
    os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_HRV_{save_date}.png"),
    dpi=400,
    bbox_inches="tight",
    facecolor="white"
)
plt.savefig(
    os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_HRV_{save_date}.pdf"),
    bbox_inches="tight"
)
plt.show()

---
#### HOBO
---

In [ ]:
#----------------------------------------
#clean and format hobo df
#----------------------------------------
hobo = hobo.iloc[:, 1:3].copy()
hobo.columns = ["timestamp", "temperature"]
hobo["temperature"] = pd.to_numeric(hobo["temperature"], errors="coerce")

# clean first
hobo = hobo.dropna(subset=["timestamp"])

# convert everything to string
ts = hobo["timestamp"].astype(str)

# detect rows that already contain a date
has_date = ts.str.contains(r"\d{2}/\d{2}/\d{2}")

# create full timestamp column
hobo["timestamp"] = pd.to_datetime(
    ts.where(has_date, date_str + " " + ts),
    errors="coerce"
)

# drop anything that still failed
hobo = hobo.dropna(subset=["timestamp"])

In [ ]:
#----------------------------------------
#Computing the hobo stats at the stations 
#----------------------------------------
def compute_station_mean_temperature(hobo_df, stations_dict):
    """
    Compute mean environmental temperature per station.

    Parameters:
    -----------
    hobo_df : pd.DataFrame
    Columns ['timestamp', 'temperature'].
    stations_dict : dict
    Dictionary with station names and (start_time, end_time) tuples.

    """

    results = []

    for station_name, (start_time, end_time) in stations_dict.items():

        
        station_temp = hobo_df[
            (hobo_df["timestamp"].dt.time >= pd.to_datetime(start_time).time()) &
            (hobo_df["timestamp"].dt.time <= pd.to_datetime(end_time).time())
        ]

        
        mean_temp = station_temp["temperature"].mean() if len(station_temp) > 0 else None

        results.append({
            "station": station_name,
            "environmental_temperature": mean_temp
        })

    return pd.DataFrame(results)

env_temp=compute_station_mean_temperature(hobo,stations)

---
#### MSR
---

In [ ]:
#----------------------------------------
#clean msr df
#----------------------------------------
msr["TIME"] = pd.to_datetime(msr["TIME"])
msr_stations = msr[(msr['TIME'] >= start_interval) & 
                   (msr['TIME'] <= end_interval)]

In [ ]:
#----------------------------------------
#Computing the msr stats at the stations 
#----------------------------------------
msr_stats_list = []

for station, (start_time, end_time) in station_windows.items():
    subset = msr[(msr['TIME'] >= start_time) & (msr['TIME'] <= end_time)]
    
    stats = {
        'station': station,
        'rh_mean': subset['RH'].mean(),
       }
    msr_stats_list.append(stats)

msr_stats = pd.DataFrame(msr_stats_list)
print(msr_stats)

---
#### iButton Chest and Hand 
---

In [ ]:
#----------------------------------------
#clean ibutton chest df
#----------------------------------------
if iButton_file_chest is not None:
    ibutton_chest['Value'] = pd.to_numeric(ibutton_chest['Value'], errors="coerce")
    ibutton_chest['Decimal'] = pd.to_numeric(ibutton_chest['Decimal'], errors="coerce")
    ibutton_chest = ibutton_chest.dropna(subset=['Value', 'Decimal'])
    ibutton_chest["Temperature"] = ibutton_chest['Value'] + ibutton_chest['Decimal'] / 1000
    ibutton_chest["DateTime"] = pd.to_datetime(ibutton_chest["Date"] + " " + ibutton_chest["Time"], format="%d.%m.%y %H:%M:%S", errors="coerce")
    ibutton_chest_plot = ibutton_chest[(ibutton_chest['DateTime'] >= start_interval) & 
                    (ibutton_chest['DateTime'] <= end_interval)]

In [ ]:
if iButton_file_chest is not None:
    temp_stats_list = []
    for station, (start_time, end_time) in station_windows.items():
        
        subset = ibutton_chest[
            (ibutton_chest['DateTime'] >= start_time) &
            (ibutton_chest['DateTime'] <= end_time)
        ]
        
        stats = {
            'station': station,
            'chest_temp_min': subset['Temperature'].min(),
            'chest_temp_max': subset['Temperature'].max(),
            'chest_temp_mean': subset['Temperature'].mean(),
            'chest_temp_std': subset['Temperature'].std(),
            'chest_temp_cv': (
                subset['Temperature'].std() / subset['Temperature'].mean()
                if subset['Temperature'].mean() != 0 else np.nan
            )
        }
        
        temp_stats_list.append(stats)
    ibutton_chest_stats = pd.DataFrame(temp_stats_list)
    print(ibutton_chest_stats)

In [ ]:
#----------------------------------------
# Plot ibutton chest temp with stations shaded
#----------------------------------------
if iButton_file_chest is not None:
    plt.figure(figsize=(14, 5))
    plt.plot(ibutton_chest_plot["DateTime"], ibutton_chest_plot["Temperature"], 
            color="blue", linewidth=0.4, label="iButtons Chest Temp")
    # Station shading 
    for start, end in station_windows_full.values():
        plt.axvspan(start, end, color="silver", alpha=0.25)

    ax = plt.gca()

    # Show only HH:MM 
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

    # X axis ticks every 10 minutes
    ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0, 60, 10)))

    plt.xticks(rotation=45, ha='right')
    plt.xlabel("Time")
    plt.ylabel("Temperature")
    plt.title(f"{save_id} iButton Device Chest Temperature over Time with Station Periods Shaded")
    plt.legend()
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    plt.tight_layout()
    plt.savefig(
        os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_ibutton_chest_{save_date}.png"),
        dpi=400,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.savefig(
        os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_ibutton_chest_{save_date}.pdf"),
        bbox_inches="tight"
    )
    plt.show()

In [ ]:
#----------------------------------------
#clean ibutton hand df
#----------------------------------------
if iButton_file_hand is not None:
    ibutton_hand['Value'] = pd.to_numeric(ibutton_hand['Value'], errors="coerce")
    ibutton_hand['Decimal'] = pd.to_numeric(ibutton_hand['Decimal'], errors="coerce")
    ibutton_hand = ibutton_hand.dropna(subset=['Value', 'Decimal'])
    ibutton_hand["Temperature"] = ibutton_hand['Value'] + ibutton_hand['Decimal'] / 1000
    ibutton_hand["DateTime"] = pd.to_datetime(ibutton_hand["Date"] + " " + ibutton_hand["Time"], format="%d.%m.%y %H:%M:%S", errors="coerce")
    ibutton_hand_plot = ibutton_hand[(ibutton_hand['DateTime'] >= start_interval) & 
                    (ibutton_hand['DateTime'] <= end_interval)]

In [ ]:
if iButton_file_hand is not None:
    temp_stats_list = []
    for station, (start_time, end_time) in station_windows.items():
        
        subset = ibutton_hand[
            (ibutton_hand['DateTime'] >= start_time) &
            (ibutton_hand['DateTime'] <= end_time)
        ]
        
        stats = {
            'station': station,
            'hand_temp_min': subset['Temperature'].min(),
            'hand_temp_max': subset['Temperature'].max(),
            'hand_temp_mean': subset['Temperature'].mean(),
            'hand_temp_std': subset['Temperature'].std(),
            'hand_temp_cv': (
                subset['Temperature'].std() / subset['Temperature'].mean()
                if subset['Temperature'].mean() != 0 else np.nan
            )
        }
        
        temp_stats_list.append(stats)
    ibutton_hand_stats = pd.DataFrame(temp_stats_list)
    print(ibutton_hand_stats)

In [ ]:
#----------------------------------------
# Plot ibutton hand temp with stations shaded
#----------------------------------------
if iButton_file_hand is not None:
    plt.figure(figsize=(14, 5))
    plt.plot(ibutton_hand_plot["DateTime"], ibutton_hand_plot["Temperature"], 
            color="blue", linewidth=0.4, label="iButtons Hand Temp")
    # Station shading 
    for start, end in station_windows_full.values():
        plt.axvspan(start, end, color="silver", alpha=0.25)

    ax = plt.gca()

    # Show only HH:MM 
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

    # X axis ticks every 10 minutes
    ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0, 60, 10)))

    plt.xticks(rotation=45, ha='right')
    plt.xlabel("Time")
    plt.ylabel("Temperature")
    plt.title(f"{save_id} iButton Device Hand Temperature over Time with Station Periods Shaded")
    plt.legend()
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    plt.tight_layout()
    plt.savefig(
        os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_ibutton_hand_{save_date}.png"),
        dpi=400,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.savefig(
        os.path.join(f'{base_dir}/{save_id}/visuals',f"{save_id}_ibutton_hand_{save_date}.pdf"),
        bbox_inches="tight"
    )
    plt.show()

In [ ]:
#----------------------------------------
# Plot ibutton hand + chest temp with stations shaded
#----------------------------------------
if iButton_file_chest is not None and iButton_file_hand is not None:
    plt.figure(figsize=(14, 5))

    # Hand
    plt.plot(
        ibutton_hand_plot["DateTime"],
        ibutton_hand_plot["Temperature"],
        color="blue",
        linewidth=0.4,
        label="iButton Hand Temp"
    )

    # Chest
    plt.plot(
        ibutton_chest_plot["DateTime"],
        ibutton_chest_plot["Temperature"],
        color="red",
        linewidth=0.4,
        label="iButton Chest Temp"
    )

    # Station shading
    for start, end in station_windows_full.values():
        plt.axvspan(start, end, color="silver", alpha=0.25)

    ax = plt.gca()

    # Show only HH:MM
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

    # X axis ticks every 10 minutes
    ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0, 60, 10)))

    plt.xticks(rotation=45, ha='right')
    plt.xlabel("Time")
    plt.ylabel("Temperature")
    plt.title(f"{save_id} iButton Temperature (Hand + Chest) with Station Periods Shaded")

    plt.legend()

    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

    plt.tight_layout()

    plt.savefig(
        os.path.join(f'{base_dir}/{save_id}/visuals', f"{save_id}_ibutton_both_{save_date}.png"),
        dpi=400,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.savefig(
        os.path.join(f'{base_dir}/{save_id}/visuals', f"{save_id}_ibutton_both_{save_date}.pdf"),
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
#----------------------------------------
#clean ibutton hand-chest df
#----------------------------------------
if iButton_file_hand is not None and iButton_file_chest is not None:
    hand_chest_merged = pd.merge(ibutton_hand[['DateTime', 'Temperature']], 
                                  ibutton_chest[['DateTime', 'Temperature']], 
                                  on='DateTime', suffixes=('_hand', '_chest'))
    hand_chest_merged['Temperature'] = hand_chest_merged['Temperature_hand'] - hand_chest_merged['Temperature_chest']

    temp_stats_list = []
    for station, (start_time, end_time) in station_windows.items():

        subset = hand_chest_merged[
            (hand_chest_merged['DateTime'] >= start_time) &
            (hand_chest_merged['DateTime'] <= end_time)
        ]

        stats = {
            'station': station,
            'hand_chest_temp_min': subset['Temperature'].min(),
            'hand_chest_temp_max': subset['Temperature'].max(),
            'hand_chest_temp_mean': subset['Temperature'].mean(),
            'hand_chest_temp_std': subset['Temperature'].std(),
            'hand_chest_temp_cv': (
                subset['Temperature'].std() / subset['Temperature'].mean()
                if subset['Temperature'].mean() != 0 else np.nan
            )
        }

        temp_stats_list.append(stats)
    ibutton_hand_chest_stats = pd.DataFrame(temp_stats_list)
    print(ibutton_hand_chest_stats)

---
#### Save Statistics
---

In [ ]:
statistics_results = hr_stats.merge(hrv_stats, on='station', how='left') \
                       .merge(actlumus_stats, on='station', how='left')

if iButton_file_hand is not None:
    statistics_results = statistics_results.merge(ibutton_hand_stats, on='station', how='left')

if iButton_file_chest is not None:
    statistics_results = statistics_results.merge(ibutton_chest_stats, on='station', how='left')

statistics_results = statistics_results.merge(env_temp, on='station', how='left') \
                                       .merge(msr_stats, on='station', how='left')

statistics_results.to_excel(f'{base_dir}/{save_id}/{save_id}_stats_results.xlsx', index=False)